# Production pipeline and prediction verification

Run from top to bottom. Quick mode is a patient subset, not final results. Notebook outputs are separate from production reports.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'features.py').exists()), None)
assert ROOT is not None, 'Open the notebook from the project repository'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_utils import *
QUICK = True  # False uses the entire eligible dataset; keep consistent across notebooks
OUT = ROOT/'reports'/('notebook_quick' if QUICK else 'notebook_full')
OUT.mkdir(parents=True, exist_ok=True)


Select existing production experiment; optionally retrain with the actual application entrypoint. This is the only notebook that launches the production training script.

In [ ]:
import subprocess,joblib,json
RUN_TRAINING=False
if RUN_TRAINING:
    subprocess.run([sys.executable,'-m','src.train_advanced',*(['--quick'] if QUICK else ['--trials','50'])],cwd=ROOT,check=True)
report=ROOT/'reports'/('advanced_quick' if QUICK else 'advanced')
modelpath=ROOT/'models'/('advanced_quick' if QUICK else 'advanced')/'best.joblib'
print('Selected experiment:',report)
if (report/'cohort.json').exists(): display(json.loads((report/'cohort.json').read_text()))

Inspect actual saved production metrics without mixing experiment scopes.

In [ ]:
if (report/'model_metrics.csv').exists():
    d=pd.read_csv(report/'model_metrics.csv'); display(d); d[d['mode'].eq('optimized')].set_index('model')[['roc_auc','pr_auc','recall']].plot.bar(figsize=(10,4)); plt.tight_layout(); plt.show()
else: print('No report yet. Enable RUN_TRAINING above or train in PowerShell.')

Verify a saved model on an eligible held-out encounter. Educational demonstration only.

In [ ]:
from src.predict import predict
if modelpath.exists():
    raw,x,y,g,parts=load_cohort(ROOT,QUICK); b=joblib.load(modelpath); row=raw.iloc[parts['test'][0]]; record={k:row[k] for k in b['raw_fields']}; display(predict(b,record))
else: print('No saved model for selected scope.')